In [1]:
# -*- coding: utf-8 -*-
"""
=============================================================================
UPI Transaction Analysis - Data Ingestion Module
=============================================================================

Description:
    This script is responsible for the extraction and ingestion of raw CSV
    data into a relational database (MySQL). It handles database connections,
    data type conversions, date parsing, and batched inserts.

Module: 3
Steps: 4 & 5
Database: MySQL

Features:
    - Smart date parsing supporting ISO and DD-MM-YYYY formats.
    - Automated foreign key checks handling.
    - Post-load row count and referential integrity validations.
    
Usage:
    Ensure `DB.env` is properly configured with your MySQL credentials before 
    running this script.
=============================================================================
"""

import pandas as pd
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
import os
import getpass


In [2]:
# 1. DATABASE CONNECTION CONFIG

In [3]:
print("\n=== Database Connection ===")
db_user = input("MySQL Username: ").strip() or "root"
db_pass = getpass.getpass("MySQL Password: ")
db_host = input("MySQL Host: ").strip() or "localhost"
db_name = input("Database: ").strip() or "upi_transactions"
csv_dir = input("CSV Folder: ").strip()
if not csv_dir:
    csv_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else '.'

DB_CONFIG = {
    "user": db_user,
    "password": db_pass,
    "host": db_host,
    "port": 3306,
    "database": db_name,
}

encoded_password = quote_plus(DB_CONFIG["password"])

connection_string = (
    f"mysql+pymysql://{DB_CONFIG['user']}:{encoded_password}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

try:
    engine = create_engine(connection_string)
    with engine.connect() as conn:
        print("Successfully connected to the database!")
except Exception as e:
    print(f"Failed to connect to the database: {e}")



=== Database Connection ===


Enter MySQL Username [default: root]:  root
Enter MySQL Host [default: localhost]:  localhost
Enter password for root@localhost:  ········

Enter the full path to the folder containing your CSV files 
[Press Enter to use the current script directory]:  C:\Users\VORTEX\OneDrive\Desktop\Data Analysis\Capstone Project\Project 5 - UPI Transactions\UPI Project Files\Dataset_Ingestions


Successfully connected to the database!


In [4]:
# 2. LOAD ORDER - parents before children, to respect FK constraints

In [5]:
load_order = [
    ("customer_master", "customer_master.csv", {"mobile_number": str}),
    ("merchant_info", "merchant_info.csv", None),
    ("device_info", "device_info.csv", None),
    ("upi_account_details", "upi_account_details.csv", None),
    ("customer_feedback_surveys", "customer_feedback_surveys.csv", None),
    ("upi_transaction_history", "upi_transaction_history.csv", None),
    ("fraud_alert_history", "fraud_alert_history.csv", None),
]

In [6]:
# 3. DATE/DATETIME COLUMNS - auto-detect ISO vs DD-MM-YYYY format

In [7]:
DATE_COLUMNS = {
    "customer_master": ["date_joined"],
    "device_info": ["last_active"],
    "upi_account_details": ["date_added"],
    "customer_feedback_surveys": ["date_submitted"],
    "upi_transaction_history": ["timestamp"],
    "fraud_alert_history": ["alert_date", "resolution_date"],
    "merchant_info": ["onboard_date"],
}

def smart_parse_dates(series, colname):
    original_na = series.isna().sum()
    parsed_iso = pd.to_datetime(series, dayfirst=False, errors="coerce")
    parsed_dmy = pd.to_datetime(series, dayfirst=True, errors="coerce")
    na_iso = parsed_iso.isna().sum() - original_na
    na_dmy = parsed_dmy.isna().sum() - original_na
    if na_iso <= na_dmy:
        if na_iso > 0:
            print(f"  WARNING: {colname} - {na_iso} unparseable dates even with ISO format")
        return parsed_iso
    else:
        print(f"  Note: {colname} detected as DD-MM-YYYY format, converted to ISO")
        return parsed_dmy

In [8]:
# 4. INGEST EACH FILE

In [9]:
print("\n---- DATA INGESTION STARTED ----")
try:
    with engine.begin() as conn:
        
        for table, csv_file, dtype in load_order:
            csv_path = os.path.join(csv_dir, csv_file)
            if not os.path.exists(csv_path):
                print(f"File not found: {csv_path}. Skipping {table}.")
                continue
                
            print(f"Reading {csv_file}...")
            df = pd.read_csv(csv_path, dtype=dtype)
            
            for col in DATE_COLUMNS.get(table, []):
                if col in df.columns:
                    df[col] = smart_parse_dates(df[col], f"{table}.{col}")
            
            print(f"Inserting into {table}...")
            # Use 'append' assuming tables are created by the DDL script
            df.to_sql(table, con=conn, if_exists="append", index=False, method="multi", chunksize=1000)
            print(f"Loaded {len(df):,} rows into {table}\n")
            
    print("Ingestion complete.")
except Exception as e:
    print(f"An error occurred during ingestion: {e}")


---- DATA INGESTION STARTED ----
Reading customer_master.csv...
Inserting into customer_master...


C:\Users\VORTEX\AppData\Local\Temp\ipykernel_28052\693796566.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed_dmy = pd.to_datetime(series, dayfirst=True, errors="coerce")


Loaded 10,000 rows into customer_master

Reading merchant_info.csv...
Inserting into merchant_info...
Loaded 500 rows into merchant_info

Reading device_info.csv...


C:\Users\VORTEX\AppData\Local\Temp\ipykernel_28052\693796566.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed_dmy = pd.to_datetime(series, dayfirst=True, errors="coerce")
C:\Users\VORTEX\AppData\Local\Temp\ipykernel_28052\693796566.py:14: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S.%f format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed_dmy = pd.to_datetime(series, dayfirst=True, errors="coerce")


Inserting into device_info...
Loaded 12,000 rows into device_info

Reading upi_account_details.csv...
Inserting into upi_account_details...


C:\Users\VORTEX\AppData\Local\Temp\ipykernel_28052\693796566.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed_dmy = pd.to_datetime(series, dayfirst=True, errors="coerce")


Loaded 12,000 rows into upi_account_details

Reading customer_feedback_surveys.csv...
Inserting into customer_feedback_surveys...


C:\Users\VORTEX\AppData\Local\Temp\ipykernel_28052\693796566.py:14: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  parsed_dmy = pd.to_datetime(series, dayfirst=True, errors="coerce")


Loaded 4,000 rows into customer_feedback_surveys

Reading upi_transaction_history.csv...
Inserting into upi_transaction_history...
Loaded 100,000 rows into upi_transaction_history

Reading fraud_alert_history.csv...
Inserting into fraud_alert_history...
Loaded 2,000 rows into fraud_alert_history

Ingestion complete.


In [10]:
# 5. POST-INGESTION VALIDATION

In [11]:
print("\n---- ROW COUNT VALIDATION ----")
with engine.connect() as conn:
    for table, _, _ in load_order:
        try:
            n = conn.execute(text(f"SELECT COUNT(*) FROM {table}")).scalar()
            print(f"{table}: {n:,} rows")
        except Exception as e:
            print(f"{table}: Could not get count ({e})")

    print("\n---- FOREIGN KEY CONSISTENCY CHECK (post-load) ----")
    fk_checks = [
        ("device_info -> customer_master",
         "SELECT COUNT(*) FROM device_info d LEFT JOIN customer_master c "
         "ON d.customer_id=c.customer_id WHERE c.customer_id IS NULL"),
        ("upi_transaction_history -> customer_master",
         "SELECT COUNT(*) FROM upi_transaction_history t LEFT JOIN customer_master c "
         "ON t.customer_id=c.customer_id WHERE c.customer_id IS NULL"),
        ("upi_transaction_history -> merchant_info (excl. NULLs)",
         "SELECT COUNT(*) FROM upi_transaction_history t LEFT JOIN merchant_info m "
         "ON t.merchant_id=m.merchant_id WHERE t.merchant_id IS NOT NULL AND m.merchant_id IS NULL"),
        ("fraud_alert_history -> upi_transaction_history",
         "SELECT COUNT(*) FROM fraud_alert_history f LEFT JOIN upi_transaction_history t "
         "ON f.transaction_id=t.transaction_id WHERE t.transaction_id IS NULL"),
    ]
    for name, q in fk_checks:
        try:
            n = conn.execute(text(q)).scalar()
            print(f"{name}: {n} orphan rows")
        except Exception as e:
            print(f"{name}: Check failed ({e})")

input("\nPress Enter to exit...")


---- ROW COUNT VALIDATION ----
customer_master: 10,000 rows
merchant_info: 500 rows
device_info: 12,000 rows
upi_account_details: 12,000 rows
customer_feedback_surveys: 4,000 rows
upi_transaction_history: 100,000 rows
fraud_alert_history: 2,000 rows

---- FOREIGN KEY CONSISTENCY CHECK (post-load) ----
device_info -> customer_master: 0 orphan rows
upi_transaction_history -> customer_master: 0 orphan rows
upi_transaction_history -> merchant_info (excl. NULLs): 0 orphan rows
fraud_alert_history -> upi_transaction_history: 0 orphan rows



Press Enter to exit... 


''